## Deep Research Experiment with `LangGraph`

This notebook aims to test multi-round research with back-and-forth between an OpenAI LLM and Perplexity. Orchestrated with `LangGraph`.



In [ ]:
import os
from datetime import datetime

from dotenv import load_dotenv
load_dotenv()

from IPython.display import display, Markdown, Image
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import START, END, StateGraph
from langgraph.constants import Send
from langgraph.graph.state import CompiledStateGraph
from langgraph.graph.message import add_messages
import operator

from pydantic import BaseModel, Field
from typing import Annotated, Sequence, TypedDict, List, Dict

import pandas as pd

### Prompts

In [ ]:
system_message_content = """You are a helpful assistant that can help with research."""
system_message = SystemMessage(content=system_message_content)

### research question writer

user_message_template = """
A user requires help with their research question. They have provided a topic they are interested in and the reason for their interest. Additionally, they have provided some background on themselves to help you understand their perspective.

<topic>
{topic}
</topic>

<reason>
{reason}
</reason>

<background>
{background}
</background>


Your task is to write research questions that can be submitted to Perplexity or Google Scholar for further research. Return a list of research questions.
"""

### answer writer prompt

answer_writer_template = """
You are assisting a user with their research. The topic is:

<topic>
{topic}
</topic>

The question you need to answer is:
<question>
{question}
</question>

Provide a detailed answer to the question.
"""

### research summarizer

research_summarizer_template = """
You are assisting a user with their research. The user has provided a research topic, for which questions have been already generated and answered. You will be provided with this list of questions and answers. Your task is to summarize the research in a professional way.

<topic>
{topic}
</topic>

<reason>
{reason}
</reason>

<background>
{background}
</background>

<questions and answers>
{questions_with_answers}
</questions and answers>

Provide the professional summary of the research.
"""

### further research validator

further_research_validator_template = """
You are assisting a user with their research. The user has provided a research topic, for which a research summary has been produced. Your task is to decide if, given the user's background and reason for interest, the summary is accurate and complete or any further deep dive is needed.

<topic>
{topic}
</topic>

<reason>
{reason}
</reason>

<background>
{background}
</background>

<research summary>
{research_summary}
</research summary>

Return with the decision if further questions are needed, a reason for the decision and if the decision is to go deeper, generate the next round of questions.
"""


### Structured Outputs

In [ ]:
class ResearchQuestion(BaseModel):
    question: str = Field(description="A research question that can be submitted to Perplexity or Google Scholar for further research.")

class ResearchQuestions(BaseModel):
    questions: List[ResearchQuestion] = Field(description="A list of research questions that can be submitted to Perplexity or Google Scholar for further research.")

class FurtherResearchValidator(BaseModel):
    decision: bool = Field(description="A decision if further questions are needed (True) or not (False).")
    reason: str = Field(description="The reason for the decision.")
    questions: List[ResearchQuestion] = Field(description="A list of research questions that can be submitted to Perplexity or Google Scholar for further research.")


### LLMs

In [ ]:
question_writer_model = "gpt-4o-2024-11-20"
further_research_validator_model = "gpt-4o-2024-11-20"
research_summarizer_model = "gpt-4o-2024-11-20"

question_writer_llm = ChatOpenAI(
    model=question_writer_model, 
    temperature=0.0
    )
question_writer_llm = question_writer_llm.with_structured_output(ResearchQuestions, method="json_schema", include_raw=True)

answer_writer_model = "llama-3.1-sonar-large-128k-online"
answer_writer_llm = ChatOpenAI(
    model=answer_writer_model, 
    temperature=0.0, 
    base_url="https://api.perplexity.ai", 
    api_key=os.getenv('PPLX_API_KEY')
    )

further_research_validator_llm = ChatOpenAI(
    model=further_research_validator_model, 
    temperature=0.0
    )
further_research_validator_llm = further_research_validator_llm.with_structured_output(FurtherResearchValidator, method="json_schema", include_raw=True)

research_summarizer_llm = ChatOpenAI(
    model=research_summarizer_model, 
    temperature=0.0
    )

### Build the graph

In [ ]:
class AgentState(TypedDict):

    topic: str
    reason: str
    background: str

    messages: Annotated[Sequence[BaseMessage], add_messages]

    questions: List[str]
    question: str
    questions_with_answers: Annotated[List[Dict[str, str]], operator.add]

    further_research_rounds: int
    further_research_needed: bool

    research_summary: str

In [ ]:
def write_questions(state: AgentState) -> AgentState:

    user_message = user_message_template.format(
        topic=state['topic'], 
        reason=state['reason'], 
        background=state['background']
        )
    
    response = question_writer_llm.invoke([system_message] + [user_message])

    response_parsed = response["parsed"]
    response_raw = response["raw"]
    response_raw.name = 'Question Writer'
    
    questions = [question.question for question in response_parsed.questions]
    
    return {
        'questions': questions,
        'messages': [system_message] + [user_message] + [response_raw]
        }

def answer_question(state: AgentState) -> AgentState:

    print(f'Answering question: {state["question"]}\n')

    user_message = answer_writer_template.format(
        topic=state['topic'], 
        question=state['question']
        )
    
    response = answer_writer_llm.invoke([system_message] + [user_message])
    response.name = 'Answer Writer'

    return {
        'messages': [response],
        'questions_with_answers': [{'question': state['question'], 'answer': response.content}]
        }

def send_questions_to_answer_writer(state: AgentState) -> AgentState:

    return [Send("answer_question", {"question": q, "topic": state['topic']}) for q in state['questions']]

def summarize_research(state: AgentState) -> AgentState:

    print('Summarizing research')

    user_message = research_summarizer_template.format(
        topic=state['topic'], 
        reason=state['reason'], 
        background=state['background'], 
        questions_with_answers=state['questions_with_answers']
        )
    
    response = research_summarizer_llm.invoke([system_message] + [user_message])
    response.name = 'Research Summarizer'

    return {
        'research_summary': response.content,
        'messages': [response]
        }


def validate_further_research(state: AgentState) -> AgentState:

    rounds_remaining = state['further_research_rounds']

    if rounds_remaining == 0:

        print('No further research rounds left.')

        return {
            'further_research_needed': False,
        }

    user_message = further_research_validator_template.format(
        topic=state['topic'], 
        reason=state['reason'], 
        background=state['background'], 
        research_summary=state['research_summary']
        )
    
    response = further_research_validator_llm.invoke([system_message] + [user_message])
    response_raw = response['raw']
    response_raw.name = 'Further Research Validator'

    if response['parsed'].decision:

        print(f'Further research needed. {rounds_remaining} rounds remaining max.')
        print(f'Reason: {response["parsed"].reason}')

        questions = [question.question for question in response['parsed'].questions]
    
        return {
            'further_research_rounds': rounds_remaining - 1,
            'further_research_needed': True,
            'questions': questions, # overwrite questions with new questions
            'messages': [response_raw]
            }
    
    else:

        print('No further research needed.')
        print(f'Reason: {response["parsed"].reason}')
        return {
            'further_research_needed': False,
            'messages': [response_raw]
            }
    
def route_from_validation(state: AgentState):
    if state['further_research_needed']:
        return "go_deeper"
    else:
        return "end"
    
def get_new_questions(state: AgentState) -> AgentState:
    pass

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("write_questions", write_questions)
workflow.add_node("answer_question", answer_question)
workflow.add_node("summarize_research", summarize_research)
workflow.add_node("validate_further_research", validate_further_research)
workflow.add_node("get_new_questions", get_new_questions)

workflow.add_edge(START, "write_questions")
workflow.add_conditional_edges("write_questions", send_questions_to_answer_writer, ["answer_question"])
workflow.add_edge("answer_question", "summarize_research")
workflow.add_edge("summarize_research", "validate_further_research")

workflow.add_conditional_edges("validate_further_research", route_from_validation, {"end": END, "go_deeper": "get_new_questions"})
workflow.add_conditional_edges("get_new_questions", send_questions_to_answer_writer, ["answer_question"])


app = workflow.compile()

In [ ]:
graph_image = app.get_graph().draw_mermaid_png()
with open("../output/research_experiment/langgraph.png", "wb") as f:
    f.write(graph_image)
display(Image(graph_image))

### Testing scenarios

In [ ]:
scenarios = [
    {
        "topic": "The GOAT of the NFL",
        "reason": "I want to know who is the GOAT of the NFL. I have an ongoing debate with a friend between Tom Brady and Patrick Mahomes.",
        "background": "I am a football fan and I have been watching the NFL for 20 years.",
        "further_research_rounds": 3,
        "further_research_needed": True
    },
    {
        "topic": "Do I invest in Bitcoin?",
        "reason": "I want to know if I should invest in Bitcoin. I have been following the news and the price has been going up since Trump's re-election.",
        "background": "I don't invest in crypto, I have some stocks and ETFs",
        "further_research_rounds": 3,
        "further_research_needed": True
    },
    {
        "topic": "Should I buy a 2yo used Tesla Model 3?",
        "reason": "I'm looking to own my first EV",
        "background": "I have never owned an EV before, if I do end up buying one, I want a Tesla",
        "further_research_rounds": 3,
        "further_research_needed": True
    }
]


In [ ]:
scenarios_results = []

for scenario in scenarios:

    print(f'Running scenario: {scenario["topic"]}')

    test = app.invoke(scenario)
    scenarios_results.append(test)
    
    print('*' * 30)

In [ ]:
for scenario in scenarios_results:
    
    # Format datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Create filename - replace spaces and special chars with underscores
    filename = f"../output/research_experiment/{scenario['topic'].replace(' ', '_').replace('?', '').replace('!', '')}_{timestamp}.md"
    
    # Write to file
    with open(filename, 'w') as f:
        f.write(scenario["research_summary"])
    
    # Still display in notebook
    display(Markdown(scenario["research_summary"]))
    print('\n' * 2)

In [ ]:
cost_mapper = {
    'gpt-4o-2024-11-20': 
        {'input_tokens' : 2.5/1000000, 'output_tokens' : 10/1000000},
    'llama-3.1-sonar-large-128k-online': 
        {'input_tokens' : 1/1000000, 'output_tokens' : 1/1000000}
}

def calculate_usage(app_result):

    usage_data = []

    for i in app_result['messages']:

        model_name = i.response_metadata if hasattr(i, 'response_metadata') else None
        try:
            model_name = model_name['model_name']
        except:
            model_name = None

        usage_metadata = i.usage_metadata if hasattr(i, 'usage_metadata') else None

        r = {
            "name" : i.name, 
            "model_name" : model_name
            }
        
        if usage_metadata:
            r.update(usage_metadata)
        
        usage_data.append(r)

    usage_df = pd.DataFrame(usage_data)
    usage_df_by_model = usage_df.groupby('model_name').agg({'input_tokens': 'sum', 'output_tokens': 'sum'}).astype(int)

    # Add cost columns
    usage_df_by_model['input_cost'] = usage_df_by_model.apply(lambda row: row['input_tokens'] * cost_mapper[row.name]['input_tokens'], axis=1)
    usage_df_by_model['output_cost'] = usage_df_by_model.apply(lambda row: row['output_tokens'] * cost_mapper[row.name]['output_tokens'], axis=1)
    usage_df_by_model['total_cost'] = usage_df_by_model['input_cost'] + usage_df_by_model['output_cost']

    print(f"Total cost: ${usage_df_by_model['total_cost'].sum():.6f}")

    return usage_df, usage_df_by_model

In [ ]:
for scenario in scenarios_results:
    print(f'Scenario: {scenario["topic"]}')
    usage, usage_by_model = calculate_usage(scenario)
    print('\n')